In [ ]:
# It looks like the pedestal file I'm using is using tile number instead of io_channel

In [ ]:
import json
import numpy as np
import tqdm
import matplotlib.pyplot as plt

In [ ]:
with open('/global/common/software/dune/inputs/FSD/pedestals/FSD_pedestals_20241112.json', 'r') as f:
    data = json.load(f)

In [ ]:
def unique_channel_id(d):
    return ((d['io_group'].astype(int)*10000+((d['io_channel'].astype(int)-1)//4)+1)*1000 \
            + d['chip_id'].astype(int))*100 + d['channel_id'].astype(int)

def unique_to_channel_id(unique):
    return unique % 100

def unique_to_chip_id(unique):
    return (unique// 100) % 1000

def unique_to_io_channel(unique):
    # Will instead extract tile number from pedestal file
    return(unique//(100*1000)) % 1000

def unique_to_tiles(unique):
    return ( (unique_to_io_channel(unique)-1) // 4) + 1

def unique_to_io_group(unique):
    return(unique // (100*1000*10000)) % 10000

In [ ]:
def convert_unique_id(unique_id):
    io_group = (unique_to_io_group(unique_id) - 1) % 4 + 1
    tile_id = (unique_to_io_channel(unique_id) - 1) % 10 + 1
    chip_id = unique_to_chip_id(unique_id)
    channel_id = unique_to_channel_id(unique_id)

    convert = (
        (io_group*10000+tile_id)*1000 \
            + chip_id)*100 + channel_id

    return convert

In [ ]:
with open('uniqueid_to_pixelid_fsd.json', 'r') as file:
    unique_id_to_pixel_id_fsd = json.load(file)

In [ ]:
pixel_ids = [] 
pedestals = []

for k, v in data.items():
    
    pedestal = v['pedestal_mv']

    unique_id = int(k)

    converted_unique_id = convert_unique_id(unique_id)
    
    pixel_id = unique_id_to_pixel_id_fsd[str(converted_unique_id)]
        
    pixel_ids.append(pixel_id)
    pedestals.append(pedestal)


out_file = 'pedestals_fsd.npz'
keys = np.array(pixel_ids, dtype='int64')
values = np.array(pedestals, dtype='float64')
default = np.array([580.])
np.savez(out_file, keys=keys, values=values, default=default)
# print(pixel_ids)

In [ ]:
len(pixel_ids)

In [ ]:
len(list(data.keys()))

In [ ]:
print( np.array(pixel_ids).min(), np.array(pixel_ids).max())